# Formula 1 Driver Performance Index (5 Seasons)

## Project aim
Build a composite **F1 Driver Performance Index** that compares drivers across the **last 5 completed seasons**.

## Notebook structure
1. Setup and imports
2. Download race and qualifying data for 5 seasons
3. Build clean event-level datasets
4. Aggregate to driver-season level
5. Handle missing data
6. Multivariate analysis
7. Normalisation
8. Weighting and aggregation
9. Compare with official standings
10. Visualisation
11. Export final outputs
12. Reflection, struggles, references, and commit log


In [18]:
import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import time

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1. Data Setup

In [19]:
BASE_URL = 'https://api.jolpi.ca/ergast/f1'
SEASONS = [2021, 2022, 2023, 2024, 2025]

DATA_DIR = Path('data')
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUT_DIR = Path('outputs')

for directory in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEASONS

[2021, 2022, 2023, 2024, 2025]

## 2. Download data

In [20]:
def fetch_json(endpoint):
    url = f"{BASE_URL}/{endpoint}"
    
    response = requests.get(url, timeout=30)
    
    if response.status_code == 429:
        print("Rate limited. Waiting 2 seconds...")
        time.sleep(2)
        response = requests.get(url, timeout=30)
    
    response.raise_for_status()
    
    # small delay to avoid hitting limit
    time.sleep(0.3)
    
    return response.json()


def get_rounds_for_season(season: int) -> list[int]:
    data = fetch_json(f'{season}.json')
    races = data['MRData']['RaceTable']['Races']
    return [int(r['round']) for r in races]

In [21]:

def build_race_rows(data: dict[str, Any]) -> list[dict[str, Any]]:
    races = data['MRData']['RaceTable']['Races']
    rows = []

    for race in races:
        for result in race.get('Results', []):
            driver = result['Driver']
            constructor = result['Constructor']

            rows.append({
                'season': int(race['season']),
                'round': int(race['round']),
                'race_name': race['raceName'],
                'date': race.get('date'),
                'driver_id': driver['driverId'],
                'driver_code': driver.get('code', ''),
                'driver_name': f"{driver['givenName']} {driver['familyName']}",
                'constructor': constructor['name'],
                'grid': pd.to_numeric(result.get('grid'), errors='coerce'),
                'finish_position': pd.to_numeric(result.get('position'), errors='coerce'),
                'points': pd.to_numeric(result.get('points'), errors='coerce'),
                'status': result.get('status', '')
            })

    return rows


def build_quali_rows(data: dict[str, Any]) -> list[dict[str, Any]]:
    races = data['MRData']['RaceTable']['Races']
    rows = []

    for race in races:
        for result in race.get('QualifyingResults', []):
            driver = result['Driver']
            constructor = result['Constructor']

            rows.append({
                'season': int(race['season']),
                'round': int(race['round']),
                'race_name': race['raceName'],
                'date': race.get('date'),
                'driver_id': driver['driverId'],
                'driver_code': driver.get('code', ''),
                'driver_name': f"{driver['givenName']} {driver['familyName']}",
                'constructor': constructor['name'],
                'quali_position': pd.to_numeric(result.get('position'), errors='coerce'),
                'q1': result.get('Q1'),
                'q2': result.get('Q2'),
                'q3': result.get('Q3')
            })

    return rows

In [22]:
all_race_rows = []
all_quali_rows = []

for season in SEASONS:
    rounds = get_rounds_for_season(season)
    print(f"Season {season}: {len(rounds)} rounds found")

    for rnd in rounds:
        race_json = fetch_json(f"{season}/{rnd}/results.json")
        quali_json = fetch_json(f"{season}/{rnd}/qualifying.json")

        all_race_rows.extend(build_race_rows(race_json))
        all_quali_rows.extend(build_quali_rows(quali_json))

race_df = pd.DataFrame(all_race_rows)
quali_df = pd.DataFrame(all_quali_rows)

race_df.to_csv(RAW_DIR / "race_results_2021_2025.csv", index=False)
quali_df.to_csv(RAW_DIR / "qualifying_results_2021_2025.csv", index=False)

print("\nRace dataset shape:", race_df.shape)
print("Qualifying dataset shape:", quali_df.shape)

print("\nRace data preview:")
display(race_df.head())

print("\nQualifying data preview:")
display(quali_df.head())

Season 2021: 22 rounds found
Season 2022: 22 rounds found
Season 2023: 22 rounds found
Season 2024: 24 rounds found
Season 2025: 24 rounds found
Rate limited. Waiting 2 seconds...
Rate limited. Waiting 2 seconds...
Rate limited. Waiting 2 seconds...
Rate limited. Waiting 2 seconds...
Rate limited. Waiting 2 seconds...

Race dataset shape: (2278, 12)
Qualifying dataset shape: (2277, 12)

Race data preview:


,season,round,race_name,date,driver_id,driver_code,driver_name,constructor,grid,finish_position,points,status
0,2021,1,Bahrain Grand Prix,2021-03-28,hamilton,HAM,Lewis Hamilton,Mercedes,2,1,25.0,Finished
1,2021,1,Bahrain Grand Prix,2021-03-28,max_verstappen,VER,Max Verstappen,Red Bull,1,2,18.0,Finished
2,2021,1,Bahrain Grand Prix,2021-03-28,bottas,BOT,Valtteri Bottas,Mercedes,3,3,16.0,Finished
3,2021,1,Bahrain Grand Prix,2021-03-28,norris,NOR,Lando Norris,McLaren,7,4,12.0,Finished
4,2021,1,Bahrain Grand Prix,2021-03-28,perez,PER,Sergio Pérez,Red Bull,0,5,10.0,Finished



Qualifying data preview:


,season,round,race_name,date,driver_id,driver_code,driver_name,constructor,quali_position,q1,q2,q3
0,2021,1,Bahrain Grand Prix,2021-03-28,max_verstappen,VER,Max Verstappen,Red Bull,1,1:30.499,1:30.318,1:28.997
1,2021,1,Bahrain Grand Prix,2021-03-28,hamilton,HAM,Lewis Hamilton,Mercedes,2,1:30.617,1:30.085,1:29.385
2,2021,1,Bahrain Grand Prix,2021-03-28,bottas,BOT,Valtteri Bottas,Mercedes,3,1:31.200,1:30.186,1:29.586
3,2021,1,Bahrain Grand Prix,2021-03-28,leclerc,LEC,Charles Leclerc,Ferrari,4,1:30.691,1:30.010,1:29.678
4,2021,1,Bahrain Grand Prix,2021-03-28,gasly,GAS,Pierre Gasly,AlphaTauri,5,1:30.848,1:30.513,1:29.809


## 3. EVENT-LEVEL FEATURE ENGINEERING

In [23]:
race_df["is_win"] = (race_df["finish_position"] == 1).astype(int)

race_df["is_podium"] = race_df["finish_position"].isin([1, 2, 3]).astype(int)

race_df["is_top10"] = race_df["finish_position"].between(1, 10, inclusive="both").fillna(False).astype(int)

race_df["is_dnf"] = (~race_df["status"].str.contains("Finished", case=False, na=False)).astype(int)

race_df["positions_gained"] = race_df["grid"] - race_df["finish_position"]

race_df.head()

,season,round,race_name,date,driver_id,driver_code,driver_name,constructor,grid,finish_position,points,status,is_win,is_podium,is_top10,is_dnf,positions_gained
0,2021,1,Bahrain Grand Prix,2021-03-28,hamilton,HAM,Lewis Hamilton,Mercedes,2,1,25.0,Finished,1,1,1,0,1
1,2021,1,Bahrain Grand Prix,2021-03-28,max_verstappen,VER,Max Verstappen,Red Bull,1,2,18.0,Finished,0,1,1,0,-1
2,2021,1,Bahrain Grand Prix,2021-03-28,bottas,BOT,Valtteri Bottas,Mercedes,3,3,16.0,Finished,0,1,1,0,0
3,2021,1,Bahrain Grand Prix,2021-03-28,norris,NOR,Lando Norris,McLaren,7,4,12.0,Finished,0,0,1,0,3
4,2021,1,Bahrain Grand Prix,2021-03-28,perez,PER,Sergio Pérez,Red Bull,0,5,10.0,Finished,0,0,1,0,-5


## 4. DRIVER-SEASON AGGREGATION

In [24]:
race_summary = (
    race_df.groupby(
        ["season", "driver_id", "driver_code", "driver_name", "constructor"],
        dropna=False
    )
    .agg(
        races=("round", "count"),
        total_points=("points", "sum"),
        points_per_race=("points", "mean"),
        average_finish_position=("finish_position", "mean"),
        finish_position_std=("finish_position", "std"),
        wins=("is_win", "sum"),
        podiums=("is_podium", "sum"),
        top10_finishes=("is_top10", "sum"),
        dnfs=("is_dnf", "sum"),
        dnf_rate=("is_dnf", "mean"),
        average_positions_gained=("positions_gained", "mean")
    )
    .reset_index()
)

race_summary["top10_rate"] = race_summary["top10_finishes"] / race_summary["races"]
race_summary["podium_rate"] = race_summary["podiums"] / race_summary["races"]
race_summary["win_rate"] = race_summary["wins"] / race_summary["races"]

print(race_summary.shape)
race_summary.head()

(113, 19)


,season,driver_id,driver_code,driver_name,constructor,races,total_points,points_per_race,average_finish_position,finish_position_std,wins,podiums,top10_finishes,dnfs,dnf_rate,average_positions_gained,top10_rate,podium_rate,win_rate
0,2021,alonso,ALO,Fernando Alonso,Alpine F1 Team,22,81.0,3.681818,9.909091,4.417596,0,1,15,11,0.500000,0.590909,0.681818,0.045455,0.000000
1,2021,bottas,BOT,Valtteri Bottas,Mercedes,22,219.0,9.954545,7.409091,6.344545,1,11,15,5,0.227273,-1.818182,0.681818,0.500000,0.045455
2,2021,gasly,GAS,Pierre Gasly,AlphaTauri,22,110.0,5.000000,9.363636,5.323313,0,1,15,9,0.409091,-3.000000,0.681818,0.045455,0.000000
3,2021,giovinazzi,GIO,Antonio Giovinazzi,Alfa Romeo,22,3.0,0.136364,13.090909,2.044949,0,0,2,18,0.818182,-0.272727,0.090909,0.000000,0.000000
4,2021,hamilton,HAM,Lewis Hamilton,Mercedes,22,385.5,17.522727,3.409091,4.349703,8,17,20,1,0.045455,-0.318182,0.909091,0.772727,0.363636


## 5. QUALIFYING AGGREGATION

In [25]:
quali_summary = (
    quali_df.groupby(
        ["season", "driver_id", "driver_code", "driver_name", "constructor"],
        dropna=False
    )
    .agg(
        average_qualifying_position=("quali_position", "mean")
    )
    .reset_index()
)

print(quali_summary.shape)
quali_summary.head()

(113, 6)


,season,driver_id,driver_code,driver_name,constructor,average_qualifying_position
0,2021,alonso,ALO,Fernando Alonso,Alpine F1 Team,11.000000
1,2021,bottas,BOT,Valtteri Bottas,Mercedes,3.772727
2,2021,gasly,GAS,Pierre Gasly,AlphaTauri,6.818182
3,2021,giovinazzi,GIO,Antonio Giovinazzi,Alfa Romeo,14.045455
4,2021,hamilton,HAM,Lewis Hamilton,Mercedes,2.136364


## 6. COMBINED DRIVER-SEASON DATASET

In [26]:
driver_season_df = race_summary.merge(
    quali_summary,
    on=["season", "driver_id", "driver_code", "driver_name", "constructor"],
    how="left"
)

driver_season_df = driver_season_df.round(4)

print(driver_season_df.shape)
driver_season_df.head()

(113, 20)


,season,driver_id,driver_code,driver_name,constructor,races,total_points,points_per_race,average_finish_position,finish_position_std,wins,podiums,top10_finishes,dnfs,dnf_rate,average_positions_gained,top10_rate,podium_rate,win_rate,average_qualifying_position
0,2021,alonso,ALO,Fernando Alonso,Alpine F1 Team,22,81.0,3.6818,9.9091,4.4176,0,1,15,11,0.5000,0.5909,0.6818,0.0455,0.0000,11.0000
1,2021,bottas,BOT,Valtteri Bottas,Mercedes,22,219.0,9.9545,7.4091,6.3445,1,11,15,5,0.2273,-1.8182,0.6818,0.5000,0.0455,3.7727
2,2021,gasly,GAS,Pierre Gasly,AlphaTauri,22,110.0,5.0000,9.3636,5.3233,0,1,15,9,0.4091,-3.0000,0.6818,0.0455,0.0000,6.8182
3,2021,giovinazzi,GIO,Antonio Giovinazzi,Alfa Romeo,22,3.0,0.1364,13.0909,2.0449,0,0,2,18,0.8182,-0.2727,0.0909,0.0000,0.0000,14.0455
4,2021,hamilton,HAM,Lewis Hamilton,Mercedes,22,385.5,17.5227,3.4091,4.3497,8,17,20,1,0.0455,-0.3182,0.9091,0.7727,0.3636,2.1364


## 7. MISSING DATA CHECK

In [27]:
missing_summary = driver_season_df.isna().sum().sort_values(ascending=False)
missing_summary

finish_position_std            3
season                         0
podiums                        0
win_rate                       0
podium_rate                    0
top10_rate                     0
average_positions_gained       0
dnf_rate                       0
dnfs                           0
top10_finishes                 0
wins                           0
driver_id                      0
average_finish_position        0
points_per_race                0
total_points                   0
races                          0
constructor                    0
driver_name                    0
driver_code                    0
average_qualifying_position    0
dtype: int64

## 7.1 HANDLE MISSING DATA


In [28]:
numeric_cols = driver_season_df.select_dtypes(include=['number']).columns

for col in numeric_cols:
    if driver_season_df[col].isna().sum() > 0:
        driver_season_df[col] = driver_season_df[col].fillna(driver_season_df[col].median())

driver_season_df.isna().sum()

season                         0
driver_id                      0
driver_code                    0
driver_name                    0
constructor                    0
races                          0
total_points                   0
points_per_race                0
average_finish_position        0
finish_position_std            0
wins                           0
podiums                        0
top10_finishes                 0
dnfs                           0
dnf_rate                       0
average_positions_gained       0
top10_rate                     0
podium_rate                    0
win_rate                       0
average_qualifying_position    0
dtype: int64

## 8. SAVE PROCESSED DATA

In [29]:
from pathlib import Path

processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

driver_season_df.to_csv(processed_dir / "driver_season_summary.csv", index=False)

print("Saved:", processed_dir / "driver_season_summary.csv")

Saved: data\processed\driver_season_summary.csv


## 9. MULTIVARIATE ANALYSIS - CORRELATION